# 06 - Modelos

Entrena modelos que aprenden de `train` y los compara contra la referencia de `05`: `mediana_4sem`, que en validación tiene **MAE 37,5** pasajeros por hora. Todo se evalúa en **validación** (octubre a diciembre de 2024, solo filas completas). Test no se toca: se usa en `07`.

| Familia | Modelos | Idea |
|---|---|---|
| `lineal` | Ridge sobre log(1 + pasajeros), sin y con interacción hora × tipo de día | El modelo más simple que puede aprender un perfil horario. La comparación entre las dos versiones mide cuánto aporta la interacción |
| `poisson` | Boosting de árboles con pérdida de Poisson | Los pasajeros son conteos y el error crece con la media (EDA). Los árboles encuentran solos las interacciones |
| `log` | Boosting de árboles sobre log(1 + pasajeros) | Misma idea que el lineal, pero sin imponer una forma |
| `residual` | Boosting de árboles que aprende la **corrección** sobre la mediana de 4 semanas | La referencia ya acierta casi siempre (correlación 0,972). Lo que falta aprender son los días en que algo cambia |

Además se recalculan tres reglas sin entrenamiento para comparar en las mismas filas: `mediana_4sem`, `perfil_historico` y `regla_por_tipo_dia` (perfil en feriados, mediana en el resto, la combinación que dejó anotada `05`).

**Cómo se elige.** Los hiperparámetros salen de una grilla chica evaluada en validación. Como el modelo ganador se elige mirando validación, su MAE ahí es algo optimista: el número limpio sale recién en test (`07`), reentrenando con `train` + `val` como indica `config.PARTICION_TRAIN_FINAL`.

**Qué se guarda.** Las métricas de cada modelo van a `resultados_modelos` con `etapa = 'modelo'`, junto a `05`, así `07` compara todo desde una sola tabla.

## 0. Preparación

In [0]:
%run ./00_setup

In [0]:
import importlib
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

from bondis import metricas, modelos, variables

for _m in (metricas, modelos, variables):
    importlib.reload(_m)

spark.conf.set("spark.sql.session.timeZone", config.TIMEZONE)
plt.rcParams.update({"figure.figsize": (13, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 200)

OBJ = modelos.OBJETIVO
REFERENCIA = "mediana_4sem"

## 1. Datos

Mismas filas que en `05`: solo `fila_completa`. Train y validación se bajan a pandas (unas 900 mil y 300 mil filas): para este tamaño scikit-learn alcanza y se evita mantener un segundo juego de métricas en Spark.

Las variables de fin de año (`es_vispera_navidad`, `es_fin_de_anio`, `es_semana_fiestas`) existen solo si gold se regeneró con el calendario nuevo. Si faltan, el notebook sigue sin ellas y lo avisa. De todos modos, **train no tiene diciembre**: esas variables valen siempre 0 en train y ningún modelo puede aprender de ellas con este corte. Eso se va a poder medir recién en test.

In [0]:
gold = variables.con_tipo_dia(spark.table(config.TABLE_GOLD)).where(F.col("fila_completa"))
leer, faltantes = modelos.columnas_a_leer(gold.columns)
if faltantes:
    print("Columnas opcionales que gold todavía no tiene (se siguen sin ellas):", faltantes)

base = gold.select(*leer, "particion")
tamanos = variables.tamano_linea(gold.where(F.col("particion") == "train")).toPandas()

train_raw = base.where(F.col("particion") == "train").drop("particion").toPandas()
val_raw = (base.where(F.col("particion") == "val").drop("particion").toPandas()
           .merge(tamanos, on="linea", how="left")
           .fillna({"tamano_linea": "chica"}))

categorias = modelos.categorias_desde(train_raw)
train = modelos.preparar(train_raw, categorias)
val = modelos.preparar(val_raw, categorias)
val["tamano_linea"] = val_raw["tamano_linea"].to_numpy()

print(f"train: {len(train):,} filas, {train['fecha'].nunique()} días, {train['linea'].nunique()} líneas")
print(f"val:   {len(val):,} filas, {val['fecha'].nunique()} días ({val['fecha'].min().date()} a {val['fecha'].max().date()})")
sin_linea = (~val["linea"].isin(categorias["linea"])).sum()
print(f"Filas de val con una línea que no está en train: {sin_linea:,}")

## 2. Referencias

Se recalculan en pandas con las mismas definiciones de `05`. El MAE de `mediana_4sem` debería dar 37,5 y el de `perfil_historico`, 37,6. Si no coinciden, algo cambió en gold desde `05`.

In [0]:
PRED = {}   # nombre del modelo -> columna de val con su predicción
for nombre, valores in modelos.referencias(train, val).items():
    val[f"pred_{nombre}"] = valores
    PRED[nombre] = f"pred_{nombre}"
    print(f"{nombre:<20} MAE {np.abs(valores - val[OBJ]).mean():6.2f}")

## 3. Modelos lineales

Ridge (regresión lineal con penalización L2) sobre log(1 + pasajeros), con estas variables:

- Logaritmo de la mediana de 4 semanas, del rezago de 1 día, del rezago de 7 días y del total de la línea el día anterior. Son las cuatro que el EDA dejó como no redundantes.
- Banderas de calendario (feriados, Carnaval, Turismo, elecciones, clases) y clima pronosticado.
- Hora, tipo de día y tipo de línea como categorías.
- Solo en la versión `interacciones`: una columna por cada par (hora, tipo de día).

`alfa` es la fuerza de la penalización: más alto, coeficientes más chicos. Se prueba una grilla corta y se elige en validación. Con 900 mil filas y menos de 200 columnas no se espera sobreajuste, así que no debería importar demasiado.

In [0]:
ALFAS = [0.1, 1, 10, 100]

CANDIDATOS = {}   # nombre -> (familia, parámetros)
for interacciones, etiqueta in ((False, "aditivo"), (True, "interacciones")):
    for alfa in ALFAS:
        CANDIDATOS[f"lineal_{etiqueta}_a{alfa}"] = ("lineal", {"interacciones": interacciones, "alfa": alfa})

ajustados = {}


def entrenar(candidatos):
    for nombre, (familia, params) in candidatos.items():
        modelo = modelos.crear_modelo(train.columns, familia, **params)
        pred, seg = modelos.ajustar_y_predecir(modelo, train, val)
        val[f"pred_{nombre}"] = pred
        PRED[nombre] = f"pred_{nombre}"
        ajustados[nombre] = modelo
        print(f"{nombre:<32} {seg:6.1f} s   MAE val {np.abs(pred - val[OBJ]).mean():6.2f}")


entrenar({n: c for n, c in CANDIDATOS.items() if c[0] == "lineal"})

## 4. Boosting de árboles

`HistGradientBoostingRegressor` construye árboles de a uno, cada uno corrigiendo los errores de los anteriores. Usa las variables sin transformar (hora, día de la semana, línea, tipo de día, los ocho rezagos, banderas y clima); `linea`, `tipo_dia` y `tipo_linea` entran como categorías.

La grilla varía el tamaño de cada árbol (`hojas`). Los demás hiperparámetros quedan fijos: 200 árboles, tasa de aprendizaje 0,1 y al menos 100 filas por hoja. No se usa parada temprana: tomaría una parte al azar de train para validar y mezclaría días futuros con pasados.

In [0]:
HOJAS = [15, 63]
for familia in ("poisson", "log", "residual"):
    for hojas in HOJAS:
        CANDIDATOS[f"hgb_{familia}_h{hojas}"] = (familia, {"hojas": hojas, "iteraciones": 200, "tasa": 0.1})

entrenar({n: c for n, c in CANDIDATOS.items() if c[0] != "lineal"})

## 5. Resultados globales en validación

Mismo método que `05`: sumas por día, métricas a partir de las sumas e intervalos del 95 % por bootstrap remuestreando **días**. Como todos los modelos se evalúan sobre los mismos días remuestreados, el intervalo de la **mejora** sobre la referencia es el que dice si la diferencia es real.

In [0]:
est = modelos.estadisticos_por_dia(val, OBJ, PRED, grupos=["tipo_dia", "tamano_linea"])

globales = metricas.agregar_mejora(metricas.metricas(est), REFERENCIA)
intervalos = metricas.bootstrap_dias(est, REFERENCIA, n_rep=1000, seed=config.SEED)
globales = globales.merge(intervalos, on="modelo").sort_values("mae").reset_index(drop=True)


def fmt_intervalo(v, inf, sup, dec=1, pct=False):
    f = (lambda x: f"{100 * x:.{dec}f} %") if pct else (lambda x: f"{x:,.{dec}f}")
    return f"{f(v)} [{f(inf)}; {f(sup)}]"


vista = pd.DataFrame({
    "modelo": globales["modelo"],
    "MAE [IC 95 %]": [fmt_intervalo(*x) for x in globales[["mae", "mae_inf", "mae_sup"]].to_numpy()],
    "RMSE": globales["rmse"].round(1),
    "R²": globales["r2"].round(3),
    "WAPE": (100 * globales["wape"]).round(1).astype(str) + " %",
    "sesgo": globales["sesgo"].round(1),
    "mejora [IC 95 %]": [fmt_intervalo(*x, pct=True) for x in globales[["mejora_mae", "mejora_mae_inf", "mejora_mae_sup"]].to_numpy()],
})
print(vista.to_string(index=False))

**Cómo leer la tabla.**
- Está ordenada por MAE. La **mejora** es 1 menos el cociente entre el MAE del modelo y el de `mediana_4sem`: 0,10 es un 10 % menos de error.
- Si el intervalo de la mejora incluye el 0, no se puede afirmar que el modelo supere a la referencia.
- Un RMSE mucho más alto que el MAE indica pocos errores muy grandes (feriados, elecciones).
- El sesgo dice hacia dónde se equivoca: positivo, se pasa; negativo, se queda corto. Los modelos con log suelen quedar con sesgo negativo, porque deshacer el logaritmo devuelve algo parecido a una mediana y no a una media.

### 5.1 Mejor modelo de cada familia

Se mira cada familia por separado para ver si la diferencia viene de la forma del modelo (lineal o árboles) o de la transformación del objetivo.

In [0]:
familia_de = {n: ("lineal_" + n.split("_")[1]) if c[0] == "lineal" else f"hgb_{c[0]}" for n, c in CANDIDATOS.items()}
cand = globales[globales["modelo"].isin(CANDIDATOS)].copy()
cand["familia"] = cand["modelo"].map(familia_de)
mejores = cand.loc[cand.groupby("familia")["mae"].idxmin()].sort_values("mae")
print(mejores[["familia", "modelo", "mae", "rmse", "sesgo", "mejora_mae", "mejora_mae_inf", "mejora_mae_sup"]]
      .round(3).to_string(index=False))

MEJOR = cand.sort_values("mae")["modelo"].iloc[0]
fila = globales.set_index("modelo").loc[MEJOR]
print(f"\nModelo elegido en validación: {MEJOR}")
print(f"  MAE {fila['mae']:.1f}, mejora sobre {REFERENCIA}: {100 * fila['mejora_mae']:.1f} % "
      f"[{100 * fila['mejora_mae_inf']:.1f} %; {100 * fila['mejora_mae_sup']:.1f} %]")
if fila["mejora_mae_inf"] <= 0:
    print("  El intervalo de la mejora incluye el 0: en validación no se despega de la referencia.")

## 6. ¿Dónde mejora y dónde no?

La mediana de 4 semanas acierta casi siempre, así que el aporte de un modelo está en los días que no se parecen a las semanas anteriores. Se comparan la referencia, las otras reglas, el mejor de cada familia y el elegido.

In [0]:
MOSTRAR = [REFERENCIA, "perfil_historico", "regla_por_tipo_dia", *mejores["modelo"]]
MOSTRAR = list(dict.fromkeys(MOSTRAR))
orden = [m for m in globales["modelo"] if m in MOSTRAR]

por_tipo = metricas.agregar_mejora(metricas.metricas(est, por=["tipo_dia"]), REFERENCIA, por=["tipo_dia"])
dias_tipo = est.drop_duplicates(["fecha", "tipo_dia"]).groupby("tipo_dia")["fecha"].nunique().rename("días")
tipos = [t for t in variables.TIPOS_DIA if t in dias_tipo.index]

print("MAE por tipo de día (pasajeros por hora):")
t = por_tipo.pivot(index="modelo", columns="tipo_dia", values="mae").loc[orden, tipos].round(1)
print(pd.concat([dias_tipo[tipos].to_frame().T, t]).to_string(), "\n")

print("Sesgo por tipo de día (> 0: se pasa):")
print(por_tipo.pivot(index="modelo", columns="tipo_dia", values="sesgo").loc[orden, tipos].round(1).to_string())

### 6.1 Por tamaño de línea

El MAE lo deciden las líneas grandes; el WAPE permite comparar grupos de distinto tamaño.

In [0]:
por_tam = metricas.metricas(est, por=["tamano_linea"])
grupos_tam = [g for g in ["grande", "mediana", "chica"] if g in set(por_tam["tamano_linea"])]
for col, titulo in (("mae", "MAE"), ("wape", "WAPE")):
    t = por_tam.pivot(index="modelo", columns="tamano_linea", values=col).loc[orden, grupos_tam]
    print(f"{titulo} por tamaño de línea:")
    print((t * 100).round(1).astype(str).add(" %").to_string() if col == "wape" else t.round(1).to_string(), "\n")

### 6.2 Día por día

MAE de cada día de validación para la referencia y el modelo elegido. Las líneas grises punteadas son días especiales; las rojas, cambios en el calendario escolar.

In [0]:
info_dias = (val[["fecha", "tipo_dia", "nombre_dia_especial", "clases_primaria", "clases_media"]]
             .drop_duplicates("fecha"))
info_dias = info_dias.assign(tipo_dia=info_dias["tipo_dia"].astype(str))

diario = metricas.metricas(est.groupby(["modelo", "fecha"], as_index=False)[metricas.ESTADISTICOS].sum(), por=["fecha"])
fig, ax = plt.subplots()
for m, color in ((REFERENCIA, "tab:gray"), (MEJOR, "tab:blue")):
    s = diario[diario["modelo"] == m].sort_values("fecha")
    ax.plot(s["fecha"], s["mae"], marker=".", ms=4, lw=1, label=m, color=color)
tope = ax.get_ylim()[1]
for _, d in info_dias.dropna(subset=["nombre_dia_especial"]).iterrows():
    ax.axvline(d["fecha"], color="gray", ls=":", lw=1)
    ax.text(d["fecha"], tope, d["nombre_dia_especial"], rotation=90, va="top", ha="right", fontsize=7)

cal = info_dias.sort_values("fecha").set_index("fecha")[["clases_primaria", "clases_media"]]
for nivel in ("primaria", "media"):
    c = cal[f"clases_{nivel}"].astype(int).diff()
    for f, v in c[c != 0].dropna().items():
        etiqueta = f"{'inicio' if v > 0 else 'fin'} de clases ({nivel})"
        ax.axvline(f, color="tab:red", ls="--", lw=1)
        ax.text(f, tope, etiqueta, rotation=90, va="top", ha="right", fontsize=7, color="tab:red")

ax.set(title="MAE por día en validación", ylabel="MAE (pasajeros por hora)")
ax.legend(loc="center left")
plt.show()

peores = (diario[diario["modelo"] == MEJOR].nlargest(10, "mae")
          .merge(info_dias, on="fecha")[["fecha", "tipo_dia", "nombre_dia_especial", "mae", "sesgo"]])
ref_dia = diario[diario["modelo"] == REFERENCIA].set_index("fecha")["mae"].rename("mae_referencia")
peores = peores.merge(ref_dia, left_on="fecha", right_index=True)
print(f"Los 10 días con más error de {MEJOR}:")
print(peores.round(1).to_string(index=False))

## 7. Curvas de entrenamiento y validación: ¿subajuste o sobreajuste?

Dos formas de equivocarse al ajustar un modelo:

- **Subajuste**: el modelo es demasiado simple y no aprende ni siquiera los datos con los que se entrena. Se ve como un error **alto en train y también alto en validación**, y agregar capacidad lo baja.
- **Sobreajuste**: el modelo memoriza detalles de train que no se repiten. Se ve como un error en train **mucho menor** que en validación, y a medida que se le da más capacidad, el de train sigue bajando mientras el de validación se estanca o sube.

**Cuidado con una trampa propia de este problema.** Validación es otro período (octubre a diciembre) y no una muestra al azar de train. Parte de la diferencia entre el error de train y el de validación viene del cambio de período y no del sobreajuste. Para separar las dos cosas se usa como termómetro a `mediana_4sem`: no aprende nada, así que su diferencia entre train y validación es puro cambio de período. Se mira el **exceso** de cada modelo, el cociente entre su brecha (MAE val / MAE train) y la de la referencia. Un exceso cerca de 1 significa que la brecha se explica por el período; bien por encima de 1, hay sobreajuste.

### 7.1 Error en train y en validación de cada candidato

El MAE de train se mide sobre una muestra de 200 mil filas.

In [0]:
N_TRAIN_CURVAS = 200_000
sub_train = train.sample(n=min(N_TRAIN_CURVAS, len(train)), random_state=config.SEED)
ref_tr = modelos.mae(sub_train[OBJ], sub_train[modelos.REFERENCIA])
ref_va = modelos.mae(val[OBJ], val[modelos.REFERENCIA])
brecha_ref = ref_va / ref_tr
print(f"{REFERENCIA}: MAE train {ref_tr:.2f}, MAE val {ref_va:.2f}, cociente val/train {brecha_ref:.2f}\n")

diag = modelos.error_train_val(ajustados, list(CANDIDATOS), train, val, n_train=N_TRAIN_CURVAS, semilla=config.SEED)
diag["brecha"] = diag["mae_val"] - diag["mae_train"]
diag["val/train"] = diag["mae_val"] / diag["mae_train"]
diag["exceso"] = diag["val/train"] / brecha_ref
diag["mejora_en_train"] = 1 - diag["mae_train"] / ref_tr
print(diag.sort_values("mae_val").round(3).to_string(index=False))

**Cómo leer la tabla.** `mejora_en_train` dice cuánto mejora el modelo sobre la referencia en los datos que vio. Si es baja, el modelo no logra ni siquiera ajustar train (subajuste, o simplemente no hay mucho más que aprender). `exceso` bien por encima de 1 señala sobreajuste. Si `exceso` es cercano a 1 y `mejora_en_train` es parecida a la mejora en validación, el modelo se comporta igual en los dos períodos.

### 7.2 Boosting: error por cantidad de árboles

Cada árbol corrige a los anteriores. La curva muestra cómo cambia el error al ir sumando árboles. Las líneas grises son el error de la referencia en cada conjunto. La línea punteada vertical marca el árbol donde el error de validación es mínimo.

- Si las dos curvas bajan juntas y siguen bajando al llegar al último árbol: **faltan árboles** (o más tasa de aprendizaje).
- Si la de train sigue bajando pero la de validación se aplana o sube: **sobreajuste**; conviene cortar antes o usar árboles más chicos.

In [0]:
ARBOLES = [n for n, c in CANDIDATOS.items() if c[0] != "lineal"]
FAMILIAS_ARBOL = ["poisson", "log", "residual"]
curvas = {n: modelos.curva_por_iteraciones(ajustados[n], train, val, n_train=100_000, semilla=config.SEED)
          for n in ARBOLES}

fig, ejes = plt.subplots(len(HOJAS), len(FAMILIAS_ARBOL), figsize=(15, 3.6 * len(HOJAS)), squeeze=False)
for i, hojas in enumerate(HOJAS):
    for j, familia in enumerate(FAMILIAS_ARBOL):
        nombre = f"hgb_{familia}_h{hojas}"
        c, ax = curvas[nombre], ejes[i][j]
        ax.plot(c["iteracion"], c["mae_train"], color="tab:blue", label="train")
        ax.plot(c["iteracion"], c["mae_val"], color="tab:orange", label="validación")
        ax.axhline(ref_tr, color="tab:blue", ls=":", lw=1, alpha=0.7)
        ax.axhline(ref_va, color="tab:orange", ls=":", lw=1, alpha=0.7)
        mejor_it = int(c.loc[c["mae_val"].idxmin(), "iteracion"])
        ax.axvline(mejor_it, color="gray", ls="--", lw=1)
        ax.set(title=f"{familia}, {hojas} hojas (mínimo de val: árbol {mejor_it})", xlabel="árboles", ylabel="MAE",
               ylim=(0, 2.0 * max(ref_tr, ref_va)))
        ax.legend()
plt.tight_layout()
plt.show()

resumen_it = pd.DataFrame([{
    "modelo": n,
    "arboles": int(c["iteracion"].iloc[-1]),
    "mejor_arbol": int(c.loc[c["mae_val"].idxmin(), "iteracion"]),
    "mae_val_minimo": c["mae_val"].min(),
    "mae_val_final": c["mae_val"].iloc[-1],
    "mae_train_final": c["mae_train"].iloc[-1],
} for n, c in curvas.items()])
resumen_it["ganancia_de_cortar"] = resumen_it["mae_val_final"] - resumen_it["mae_val_minimo"]
print(resumen_it.round(3).to_string(index=False))

Si `mejor_arbol` es igual a `arboles`, el error de validación seguía bajando al terminar: se puede probar con más árboles. Si es mucho menor, los árboles de más solo sobreajustan. `ganancia_de_cortar` es cuánto MAE se ahorraría usando el árbol óptimo; si es chica, el número de árboles elegido está bien.

### 7.3 Modelos lineales: error según `alfa`

`alfa` es la penalización de Ridge. Con `alfa` muy grande el modelo queda demasiado rígido (subajuste). Con `alfa` muy chico se parece a una regresión sin penalización y puede sobreajustar, aunque con tantas filas y pocas columnas casi no pasa.

In [0]:
lin = diag[diag["modelo"].isin([n for n, c in CANDIDATOS.items() if c[0] == "lineal"])].copy()
lin["alfa"] = lin["modelo"].map(lambda n: CANDIDATOS[n][1]["alfa"])
lin["variante"] = lin["modelo"].map(lambda n: "con interacciones" if CANDIDATOS[n][1]["interacciones"] else "aditivo")

fig, ax = plt.subplots(figsize=(8, 4))
for (variante, g), color in zip(lin.groupby("variante"), ("tab:green", "tab:purple")):
    g = g.sort_values("alfa")
    ax.plot(g["alfa"], g["mae_train"], ls="--", marker="o", color=color, label=f"{variante}, train")
    ax.plot(g["alfa"], g["mae_val"], ls="-", marker="o", color=color, label=f"{variante}, validación")
ax.axhline(ref_va, color="gray", ls=":", lw=1, label=f"{REFERENCIA}, validación")
ax.set(xscale="log", xlabel="alfa", ylabel="MAE", title="Ridge: error según la penalización")
ax.legend(fontsize=8)
plt.show()
print(lin.sort_values(["variante", "alfa"])[["variante", "alfa", "mae_train", "mae_val", "brecha", "exceso"]].round(3).to_string(index=False))

### 7.4 Curva de aprendizaje del modelo elegido: error según los datos de entrenamiento

Se vuelve a entrenar el modelo elegido con una fracción cada vez mayor de los **días** de train (se sortean días completos, no filas) y se mide el error en esos mismos datos y en validación.

- Las dos curvas se juntan y quedan **planas y altas**: el modelo ya no mejora con más datos; el problema es el modelo (sesgo, subajuste) o el ruido propio del problema.
- Hay una **brecha que se achica** al agregar datos: más datos ayudarían, y el modelo hoy está algo sobreajustado.
- Cuidado: con pocos días, train no ve todos los tipos de día. La brecha del extremo izquierdo mezcla sobreajuste con falta de ejemplos.

In [0]:
familia_m, params_m = CANDIDATOS[MEJOR]
curva_ap = modelos.curva_aprendizaje(
    lambda: modelos.crear_modelo(train.columns, familia_m, **params_m),
    train, val, fracciones=(0.1, 0.25, 0.5, 0.75, 1.0), semilla=config.SEED)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curva_ap["dias"], curva_ap["mae_train"], marker="o", color="tab:blue", label="train")
ax.plot(curva_ap["dias"], curva_ap["mae_val"], marker="o", color="tab:orange", label="validación")
ax.axhline(ref_tr, color="tab:blue", ls=":", lw=1, alpha=0.7, label=f"{REFERENCIA}, train")
ax.axhline(ref_va, color="tab:orange", ls=":", lw=1, alpha=0.7, label=f"{REFERENCIA}, validación")
ax.set(xlabel="días de train usados", ylabel="MAE", title=f"Curva de aprendizaje: {MEJOR}")
ax.legend(fontsize=8)
plt.show()
print(curva_ap.round(3).to_string(index=False))

**Lectura de la sección 7.** Completar en las conclusiones qué muestran las curvas: si algún modelo sobreajusta (exceso bien por encima de 1, validación que sube al agregar árboles), si hay subajuste (error alto en train y mejora baja sobre la referencia) y si más datos ayudarían. Con solo 2024 en train, tres de los cuatro trimestres del año quedan sin ver; eso limita lo que cualquier modelo puede aprender y se resuelve en parte al reentrenar con train + val.

## 8. Importancia de las variables

Importancia por **permutación**: se desordenan las filas de un grupo de variables en una muestra de validación y se mide cuánto sube el MAE. Si sube mucho, el modelo dependía de ese grupo. Las variables se agrupan porque varias dicen casi lo mismo (los rezagos, el promedio y la mediana de 4 semanas), y desordenar una sola sería compensado por las otras.

Es una lectura aproximada: no mide efectos causales, y un grupo con importancia baja puede ser irrelevante o estar replicado por otro.

In [0]:
muestra = val.sample(n=min(100_000, len(val)), random_state=config.SEED)
imp = modelos.importancia_por_grupos(ajustados[MEJOR], muestra, modelos.grupos_de_variables(val.columns),
                                     n_rep=3, seed=config.SEED)
print(f"MAE de {MEJOR} en la muestra sin desordenar: {imp.attrs['mae_base']:.2f}\n")
print(imp[["grupo", "mae_permutado", "aumento_mae", "aumento_pct"]].round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(imp["grupo"][::-1], imp["aumento_mae"][::-1], color="tab:blue")
ax.set(title=f"Aumento del MAE al desordenar cada grupo ({MEJOR})", xlabel="aumento del MAE (pasajeros por hora)")
plt.show()

## 9. Guardar los resultados

Se guardan en `resultados_modelos` con `etapa = 'modelo'`, junto a las líneas base de `05`. Re-ejecutar este notebook reemplaza solo las filas de la etapa `modelo`. Se agregan dos columnas: `parametros` (los hiperparámetros de cada modelo) y `seleccionado` (cuál fue el elegido).

In [0]:
ETAPA = "modelo"
salida = globales[globales["modelo"].isin(CANDIDATOS)].copy()
salida = salida.assign(
    etapa=ETAPA, conjunto="val", referencia=REFERENCIA,
    parametros=[f"{CANDIDATOS[m][0]} {CANDIDATOS[m][1]}" for m in salida["modelo"]],
    seleccionado=(salida["modelo"] == MEJOR).to_numpy(),
    fecha_ejecucion=datetime.now().strftime("%Y-%m-%d %H:%M"),
)
columnas = ["etapa", "conjunto", "modelo", "referencia", "filas", "mae", "mae_inf", "mae_sup", "rmse",
            "rmse_inf", "rmse_sup", "r2", "wape", "sesgo", "mejora_mae", "mejora_mae_inf", "mejora_mae_sup",
            "parametros", "seleccionado", "fecha_ejecucion"]
no_numericas = ("etapa", "conjunto", "modelo", "referencia", "fecha_ejecucion", "filas", "parametros", "seleccionado")
salida = salida[columnas].astype({c: "float64" for c in columnas if c not in no_numericas})

escritor = spark.createDataFrame(salida).write.format("delta").mode("overwrite").option("mergeSchema", "true")
if spark.catalog.tableExists(config.TABLE_RESULTADOS):
    escritor.option("replaceWhere", f"etapa = '{ETAPA}'").saveAsTable(config.TABLE_RESULTADOS)
else:
    escritor.saveAsTable(config.TABLE_RESULTADOS)

print(spark.table(config.TABLE_RESULTADOS).where(F.col("etapa") == ETAPA)
      .select("modelo", "mae", "rmse", "r2", "mejora_mae", "seleccionado").toPandas().round(3).to_string(index=False))

## Conclusiones

Completar con los resultados de la ejecución, como en `05`. Resultados en validación (01/10/2024 a 31/12/2024, solo filas completas).

| Pregunta | Resultado |
|---|---|
| ¿Qué modelo tiene menos error? ¿Qué familia es? | |
| ¿Cuánto mejora sobre `mediana_4sem` (MAE 37,5)? ¿El intervalo incluye el 0? | |
| ¿La interacción hora × tipo de día mejora al modelo lineal? ¿Cuánto? | |
| ¿Qué transformación del objetivo funciona mejor en los árboles (Poisson, log, residual)? | |
| ¿El modelo elegido le gana a `regla_por_tipo_dia`? Si no, ¿qué aporta entrenar? | |
| ¿En qué tipo de día mejora más? ¿Y en cuál empeora? | |
| ¿Mejora en las líneas grandes, medianas y chicas por igual? | |
| ¿Qué grupo de variables pesa más? ¿Hay alguno que no aporta? | |
| ¿Hay subajuste o sobreajuste? ¿Cuántos árboles conviene usar? ¿Más datos ayudarían? | |
| ¿Qué errores quedan sin resolver (días especiales, elecciones, diciembre)? | |

**Limitaciones.** El modelo se elige con validación, así que su MAE ahí es algo optimista. Train no tiene diciembre: las variables de fin de año no se pueden aprender con este corte y los feriados y elecciones de train son muy pocos. En `07` se reentrena con `train` + `val` (todo 2024) y se evalúa en test.